# Tesseract OCR on Whole Image (Rearranged)

Target image: `Icdar2013\\Challenge2_Test_Task12_Images\\img_1.jpg`

## Shared: Read Ground Truth

**Syntax:** `Path.read_text(encoding, errors)`

**Intro:** Load ground truth text from file for comparison. Uses `pathlib.Path` to read UTF-8 encoded text files.

In [ ]:
from pathlib import Path
gt_path = Path(r"Icdar2013\\Challenge2_Test_Task1_GT (1)\\gt_img_1.txt")
gt_text = gt_path.read_text(encoding="utf-8", errors="ignore").strip()
print("Ground truth loaded.")
print(f"- {gt_path.resolve()}")

## 1. Apply on Original Image

### 1) Initialize Tesseract

**Syntax:** `pytesseract.pytesseract.tesseract_cmd = r"path\\to\\tesseract.exe"`

**Intro:** Configure the path to Tesseract OCR executable. Required so PyTesseract knows where to find the OCR engine on your system.

In [ ]:
import cv2
import pytesseract

# Set this if Tesseract is not in PATH
pytesseract.pytesseract.tesseract_cmd = r"C:\\Program Files\\Tesseract-OCR\\tesseract.exe"

# OCR_LANG = "eng"
# OCR_CONFIG = "--oem 3 --psm 6"

### 2) Read image

**Syntax:** `cv2.imread(filename) -> numpy.ndarray`

**Intro:** Load image from disk using OpenCV. Returns BGR image as NumPy array. Check `.shape` to verify dimensions (height, width, channels).

In [ ]:
# image_path = Path(r"Icdar2013\\Challenge2_Test_Task12_Images\\img_1.jpg")
image_path = Path(r"../../Icdar2013/Challenge2_Test_Task12_Images/img_1.jpg")
img_bgr = cv2.imread(str(image_path))
print(f"Loaded {image_path.resolve()}")
print(f"Image shape: {img_bgr.shape}")

### 3) Apply OCR

**Syntax:** `pytesseract.image_to_string(image, lang="eng", config="--oem 3 --psm 6") -> str`

**Intro:** Extract text from image using Tesseract OCR. Returns detected text as string. Can use optional config flags for engine mode (oem) and page segmentation mode (psm).

In [ ]:
# detected = pytesseract.image_to_string(img_bgr, lang="eng", config="--oem 3 --psm 6")
detected = pytesseract.image_to_string(img_bgr, lang="eng")
print(f"OCR Result:\n{detected}")

### 4) Calculate CER and WER

**Syntax:** `levenshtein_distance(seq1, seq2) -> int` | CER = distance / ref_chars | WER = distance / ref_words

**Intro:** Compute edit distance between ground truth and predicted text. CER (Character Error Rate) and WER (Word Error Rate) measure OCR accuracy. Lower is better.

In [ ]:
def levenshtein_distance(seq1, seq2):
    n, m = len(seq1), len(seq2)
    if n == 0: return m
    if m == 0: return n
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if seq1[i - 1] == seq2[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )
    return dp[n][m]

pred_text_original = detected.strip()
gt_norm = gt_text.strip()
char_dist = levenshtein_distance(list(gt_norm), list(pred_text_original))
cer = char_dist / max(1, len(gt_norm))
gt_words = gt_norm.split()
pred_words = pred_text_original.split()
word_dist = levenshtein_distance(gt_words, pred_words)
wer = word_dist / max(1, len(gt_words))
metrics_original = {
    "cer": cer,
    "wer": wer,
    "char_distance": char_dist,
    "word_distance": word_dist,
    "gt_num_chars": len(gt_norm),
    "pred_num_chars": len(pred_text_original),
    "gt_num_words": len(gt_words),
    "pred_num_words": len(pred_words)
}
print("Original metrics:")
print(metrics_original)

## 2. Optimized with Preprocessed

### 1) Preprocessed

**Syntax:** `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)` | `cv2.GaussianBlur(img, (k,k), 0)` | `cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)`

**Intro:** Prepare image for OCR by (1) converting to grayscale, (2) applying Gaussian blur to reduce noise, (3) binarizing with Otsu threshold to isolate text from background.

In [ ]:
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (3, 3), 0)
_, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

**Syntax:** `matplotlib.pyplot.imshow(img, cmap=...)` | `ax.set_title(...)` | `ax.axis("off")`

**Intro:** Display preprocessing steps side-by-side for visual inspection. Shows original BGR, grayscale, and binary images.

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
ax1.imshow(img_bgr)
ax1.set_title("Original Image")
ax1.axis("off")
ax2.imshow(gray, cmap="gray")
ax2.set_title("Grayscale Image")
ax2.axis("off")
ax3.imshow(binary, cmap="gray")
ax3.set_title("Binary Image (Otsu Threshold)")
ax3.axis("off")

### 2) Apply OCR

**Syntax:** `pytesseract.image_to_data(image, lang="eng", output_type=pytesseract.Output.DATAFRAME) -> DataFrame`

**Intro:** Run OCR on preprocessed binary image and return token-level data (text, confidence, bounding boxes) as DataFrame. More detailed than raw string output.

In [ ]:
# deteced_preprocessed = pytesseract.image_to_string(binary, lang="eng", config="--oem 3 --psm 6")
deteced_preprocessed = pytesseract.image_to_string(binary, lang="eng")
print("Raw OCR preview:\n")
print(deteced_preprocessed[:1000])

# Optional data output for filtering
data = pytesseract.image_to_data(binary, lang="eng", output_type=pytesseract.Output.DATAFRAME)

### 3) Calculate CER and WER

**Syntax:** `df.dropna(subset=[...])` | `df[df["conf"] >= 60]` | `" ".join(tokens)`

**Intro:** Filter OCR tokens by confidence threshold (≥60), remove blanks, and join into clean text. Then compute CER/WER same as original pipeline.

In [ ]:
result_preprocessed = deteced_preprocessed.strip()
print(result_preprocessed)

df = data.copy()
df = df.dropna(subset=["text", "conf"])
df = df[df["text"].astype(str).str.strip() != ""]
df = df[df["conf"] >= 60]

clean_text = " ".join(df["text"].astype(str).tolist()).strip()
if not clean_text:
    clean_text = result_preprocessed.strip()

pred_text = clean_text.strip()
char_dist = levenshtein_distance(list(gt_norm), list(pred_text))
cer = char_dist / max(1, len(gt_norm))
pred_words = pred_text.split()
word_dist = levenshtein_distance(gt_words, pred_words)
wer = word_dist / max(1, len(gt_words))

metrics_preprocessed = {
    "cer": cer,
    "wer": wer,
    "char_distance": char_dist,
    "word_distance": word_dist,
    "gt_num_chars": len(gt_norm),
    "pred_num_chars": len(pred_text),
    "gt_num_words": len(gt_words),
    "pred_num_words": len(pred_words)
}

print("Preprocessed metrics:")
print(metrics_preprocessed)

## Save outputs

**Syntax:** `Path.mkdir(exist_ok=True)` | `Path.write_text(content, encoding="utf-8")`

**Intro:** Create output directory and save OCR text to `.txt` file.

In [ ]:
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

ocr_out_path = out_dir / "img_1_ocr.txt"
ocr_out_path.write_text(clean_text, encoding="utf-8")
print(f"OCR saved: {ocr_out_path.resolve()}")

**Syntax:** `json.dumps(dict, indent=2)` | `Path.write_text(...)`

**Intro:** Serialize metrics dictionary to JSON and save to file for results tracking and comparison.

In [ ]:
import json
metrics_path = out_dir / "img_1_metrics.json"
all_metrics = {
    "original": metrics_original,
    "preprocessed": metrics_preprocessed
}
metrics_path.write_text(json.dumps(all_metrics, indent=2), encoding="utf-8")
print(f"Metrics saved: {metrics_path.resolve()}")